# **Smarter Search with Advanced LangChain Retrieval**


## __Table of Contents__

<ol>
    <li><a href="#Overview">Overview</a></li>
    <li><a href="#Objectives">Objectives</a></li>
    <li>
        <a href="#Setup">Setup</a>
        <ol>
            <li><a href="#Installing-required-libraries">Installing required libraries</a></li>
            <li><a href="#Defining-helper-functions">Defining helper functions</a></li>
        </ol>
    </li>
    <li><a href="#Creating-a-retriever-model">Creating a retriever model</a></li>
    <ol>
        <li><a href="#Build-the-LLM">Build the LLM</a></li>
        <li><a href="#Use-the-text-splitter">Use the text splitter</a></li>
        <li><a href="#Create-the-embedding-model">Create the embedding model</a></li>
        <li>
            <a href="#Use-Retrievers">Use Retrievers</a>
            <ol>
                <li><a href="#Vector-Store-Backed-Retriever">Vector Store-Backed Retriever</a></li>
                <li><a href="#Multi-Query-Retriever">Multi-Query Retriever</a></li>
                <li><a href="#Self-Querying-Retriever">Self-Querying Retriever</a></li>
                <li><a href="#Parent-Document-Retriever">Parent Document Retriever</a></li>
            </ol>
        </li>
    </ol>

   
            
<li><a href="#Exercises">Exercises</a>
<ol>
<li><a href="#Retrieve-Top-2-Results-Using-a-Vector-Store-Backed-Retriever">Retrieve Top 2 Results Using Vector Store-Backed Retriever</a></li>
<li><a href="#Self-Querying-Retriever-for-a-Query">Self-Querying Retriever for a Query</a></li>
</ol>
</li>


## Overview


Imagine you are working on a project that involves processing a large collection of text documents, such as research papers, legal documents, or customer service logs. Your task is to develop a system that can quickly retrieve the most relevant segments of text based on a user's query. Traditional keyword-based search methods might not be sufficient, as they often fail to capture the nuanced meanings and contexts within the documents. To address this challenge, you can use different types of retrievers based on LangChain.

Using retrievers is crucial for several reasons:

- **Efficiency:** Retrievers enable fast and efficient retrieval of relevant information from large datasets, saving time and computational resources.
- **Accuracy:** By leveraging advanced retrieval techniques, these tools can provide more accurate and contextually relevant results compared to traditional search methods.
- **Versatility:** Different retrievers can be tailored to specific use cases, making them adaptable to various types of text data and query requirements.
- **Context awareness:** Some retrievers, such as the Parent Document Retriever, can consider the broader context of the document, enhancing the relevance of the retrieved segments.


We will learn about four types of retrievers: `Vector Store-backed Retriever`, `Multi-Query Retriever`, `Self-Querying Retriever`, and `Parent Document Retriever`.

## Objectives

After completing this lab, you will be able to:

- Use various types of retrievers to efficiently extract relevant document segments from text, leveraging LangChain's capabilities.
- Apply the Vector Store-backed Retriever to solve problems involving semantic similarity and relevance in large text datasets.
- Utilize the Multi-Query Retriever to address situations where multiple query variations are needed to capture comprehensive results.
- Implement the Self-Querying Retriever to automatically generate and refine queries, enhancing the accuracy of information retrieval.
- Employ the Parent Document Retriever to maintain context and relevance by considering the broader context of the parent document.


----


## Setup


For this lab, you will use the following libraries:

*   `google-genai` for using LLMs from Google AI.
*   [`langchain`, `langchain-ibm`, `langchain-community`](https://www.langchain.com/) for using relevant features from LangChain.
*   [`pypdf`](https://pypi.org/project/pypdf/)is an open-source pure Python PDF library capable of splitting, merging, cropping, and transforming the pages of PDF files.
*   [`chromadb`](https://www.trychroma.com/) is an open-source vector database used to store embeddings.
*   [`lark`](https://pypi.org/project/lark/) is a general-purpose parsing library for Python. It is necessary for a Self-Querying Retriever.


### Installing required libraries

The following required libraries are __not__ preinstalled in the Skills Network Labs environment. __You must run the following cell__ to install them:

**Note:** The version is being pinned here to specify the version. It's recommended that you do this as well. Even if the library is updated in the future, the installed library could still support this lab work.

This might take approximately 1-2 minutes.


In [2]:
%pip install -U \
    langchain \
    langchain-community \
    langchain-google-genai \
    chromadb \
    pypdf \
    sentence-transformers \
    lark | tail -n 1
%pip install "sentence-transformers" | tail -n 1
%pip install "huggingface-hub" | tail -n 1
%pip install "posthog" | tail -n 1
%pip install "langchain-chroma" | tail -n 1


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


After you install the libraries, restart your kernel. You can do that by clicking the **Restart the kernel** icon.

<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/QrUNwLZfVySxQ9xvbOJgyQ/restart.png" width="80%" alt="Restart kernel">


## Defining helper functions

Use the following code to define some helper functions to reduce the repeat work in the notebook:


In [3]:
# You can use this section to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

## Creating a retriever model


The following steps are involved  to create a retriever model using LangChain:

- Building LLMs
  
- Splitting documents into chunks
  
- Building an embedding model
  
- Retrieving related knowledge from text
  


### Build the LLM
Develop or select a pre-trained language model that can understand and generate human-like text. This model serves as the foundation for processing and interpreting language data.


In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI

# Set the environment variable for the Google API key
os.environ["GOOGLE_API_KEY"] = "Dummy API Key"

The following will allow us to connect to watsonx.ai and set LLM parameters


In [5]:

def llm():
    gemini_llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0.5,
        max_tokens=256
    )

    return gemini_llm

### Use the text splitter
Break down large documents into smaller, manageable pieces or chunks. This helps in processing and analyzing the text more efficiently, allowing the model to focus on specific sections rather than being overwhelmed by the entire document.


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [7]:
def text_splitter(data, chunk_size, chunk_overlap):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
    )
    chunks = text_splitter.split_documents(data)
    return chunks

### Create the embedding model


Create or utilize an embedding model to convert chunks of text into numerical vectors. These vectors represent the semantic meaning of the text, enabling the model to compare and retrieve relevant information based on similarity.
Here we will be using HuggingFace model for the Embeddings.

In [8]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [9]:
def hf_embedding():
    embedding_model = HuggingFaceEmbeddings(
        model_name="BAAI/bge-base-en-v1.5"
    )
    return embedding_model

### Use Retrievers

A retriever is an interface designed to return documents based on an unstructured query. Unlike a vector store, which stores and retrieves documents, a retriever's primary function is to find and return relevant documents. While vector stores can serve as the backbone of a retriever, there are various other types of retrievers that can be used as well.

Retrievers take a string `query` as input and output a list of `Documents`.


#### Vector Store-Backed Retriever


A vector store retriever is a type of retriever that utilizes a vector store to fetch documents. It acts as a lightweight wrapper around the vector store class, enabling it to conform to the retriever interface. This retriever leverages the search methods implemented by the vector store, such as similarity search and Maximum Marginal Relevance (MMR), to query texts stored within it.


Before demonstrating this retriever, you need to load some example text. A `.txt` document has been prepared for you.


In [10]:
!curl -o "companypolicies.txt" "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/MZ9z1lm-Ui3YBp3SYWLTAQ/companypolicies.txt"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 15660  100 15660    0     0   4277      0  0:00:03  0:00:03 --:--:--  4277


Use `TextLoader` to load the document.


In [11]:
from langchain_community.document_loaders import TextLoader

In [12]:
loader = TextLoader("companypolicies.txt")
txt_data = loader.load()

Let's take a look at this document. This is a document about different policies in a company.


In [13]:
txt_data

[Document(metadata={'source': 'companypolicies.txt'}, page_content="1.\tCode of Conduct\n\nOur Code of Conduct outlines the fundamental principles and ethical standards that guide every member of our organization. We are committed to maintaining a workplace that is built on integrity, respect, and accountability.\nIntegrity: We hold ourselves to the highest ethical standards. This means acting honestly and transparently in all our interactions, whether with colleagues, clients, or the broader community. We respect and protect sensitive information, and we avoid conflicts of interest.\nRespect: We embrace diversity and value each individual's contributions. Discrimination, harassment, or any form of disrespectful behavior is unacceptable. We create an inclusive environment where differences are celebrated and everyone is treated with dignity and courtesy.\nAccountability: We take responsibility for our actions and decisions. We follow all relevant laws and regulations, and we strive to 

Split `txt_data` into chunks. `chunk_size = 200`, `chunk_overlap = 20` has been set.


In [14]:
chunks_txt = text_splitter(txt_data, 200, 20)

Store the embeddings into a `ChromaDB`.


In [15]:
from langchain_chroma import Chroma

In [16]:
vectordb = Chroma.from_documents(chunks_txt, hf_embedding())

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6941.56it/s]


##### Simple similarity search


Here is an example of a simple similarity search based on the vector database.

For this demonstration, the query has been set to "email policy".


In [17]:
query = "email policy"
retriever = vectordb.as_retriever()

In [18]:
docs = retriever.invoke(query)

By default, the number of retrieval results is four, and they are ranked by similarity level.


In [19]:
docs

[Document(id='8ed1caca-bb5c-4cd9-9c57-39a49124d3a2', metadata={'source': 'companypolicies.txt'}, page_content='3.\tInternet and Email Policy'),
 Document(id='72b2ef42-5ff3-4787-9969-5c8f2ebcdbd0', metadata={'source': 'companypolicies.txt'}, page_content='Harassment and Inappropriate Content: Internet and email usage must not involve harassment, discrimination, or the distribution of offensive or inappropriate content. Show respect and sensitivity to'),
 Document(id='4927366d-ac37-4eae-bbda-183f2fee056f', metadata={'source': 'companypolicies.txt'}, page_content='Our Internet and Email Policy aims to promote safe, responsible usage of digital communication tools that align with our values and legal obligations. Each employee is expected to understand and'),
 Document(id='8b96e15c-68da-4582-85d4-ad56b15c9767', metadata={'source': 'companypolicies.txt'}, page_content='Confidentiality: Reserve email for the transmission of confidential information, trade secrets, and sensitive customer data

You can also specify `search kwargs` like `k` to limit the retrieval results.


In [20]:
retriever = vectordb.as_retriever(search_kwargs={"k": 1})
docs = retriever.invoke(query)
docs

[Document(id='8ed1caca-bb5c-4cd9-9c57-39a49124d3a2', metadata={'source': 'companypolicies.txt'}, page_content='3.\tInternet and Email Policy')]

##### MMR search


MMR in vector stores is a technique used to balance the relevance and diversity of retrieved results. It selects documents that are both highly relevant to the query and minimally similar to previously selected documents. This approach helps to avoid redundancy and ensures a more comprehensive coverage of different aspects of the query.


The following code is showing how to conduct an MMR search in a vector database. You just need to sepecify `search_type="mmr"`.


In [21]:
retriever = vectordb.as_retriever(search_type="mmr")
docs = retriever.invoke(query)
docs

[Document(id='8ed1caca-bb5c-4cd9-9c57-39a49124d3a2', metadata={'source': 'companypolicies.txt'}, page_content='3.\tInternet and Email Policy'),
 Document(id='8b96e15c-68da-4582-85d4-ad56b15c9767', metadata={'source': 'companypolicies.txt'}, page_content='Confidentiality: Reserve email for the transmission of confidential information, trade secrets, and sensitive customer data only when encryption is applied. Exercise discretion when discussing'),
 Document(id='81e52c25-d1e5-419f-915d-76f840e72fc7', metadata={'source': 'companypolicies.txt'}, page_content='7.\tHealth and Safety Policy'),
 Document(id='048859fa-082c-45b4-b354-7ee3008254b6', metadata={'source': 'companypolicies.txt'}, page_content='form of unlawful bias. This policy applies to every individual within the organization, including employees, contractors, visitors, and clients.')]

##### Similarity score threshold retrieval


You can also set a retrieval method that defines a similarity score threshold, returning only documents with a score above that threshold.


In [22]:
retriever = vectordb.as_retriever(
    search_type="similarity_score_threshold", search_kwargs={"score_threshold": 0.4}
)
docs = retriever.invoke(query)
docs

[Document(id='8ed1caca-bb5c-4cd9-9c57-39a49124d3a2', metadata={'source': 'companypolicies.txt'}, page_content='3.\tInternet and Email Policy'),
 Document(id='72b2ef42-5ff3-4787-9969-5c8f2ebcdbd0', metadata={'source': 'companypolicies.txt'}, page_content='Harassment and Inappropriate Content: Internet and email usage must not involve harassment, discrimination, or the distribution of offensive or inappropriate content. Show respect and sensitivity to'),
 Document(id='4927366d-ac37-4eae-bbda-183f2fee056f', metadata={'source': 'companypolicies.txt'}, page_content='Our Internet and Email Policy aims to promote safe, responsible usage of digital communication tools that align with our values and legal obligations. Each employee is expected to understand and'),
 Document(id='8b96e15c-68da-4582-85d4-ad56b15c9767', metadata={'source': 'companypolicies.txt'}, page_content='Confidentiality: Reserve email for the transmission of confidential information, trade secrets, and sensitive customer data

#### Multi-Query Retriever


Distance-based vector database retrieval represents queries in high-dimensional space and finds similar embedded documents based on "distance". However, retrieval results may vary with subtle changes in query wording or if the embeddings do not accurately capture the data's semantics.

The `MultiQueryRetriever` addresses this by using an LLM to generate multiple queries from different perspectives for a given user input query. For each query, it retrieves a set of relevant documents and then takes the unique union of these results to form a larger set of potentially relevant documents. By generating multiple perspectives on the same question, the `MultiQueryRetriever` can potentially overcome some limitations of distance-based retrieval, resulting in a richer and more diverse set of results.


The following picture shows the difference between retrievers solely based on distance and the Multi-Query Retriever.


<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/NCZCJ26bp3uKTa0gp8Agwg/multiquery.png" width="90%" alt="multiquery"/>


Let's consider the query sentence, `"I like cats"`.

On the upper side of the picture, you can see a retriever that relies solely on distance. This retriever calculates the distance between the query and the documents in the vector store, returning the document with the closest match.

On the lower side, you can see a multi-query retriever. It first uses an LLM to generate multiple queries from different perspectives based on the user's input query. For each generated query, it retrieves relevant documents and then returns the union of these results.


A PDF document has been prepared to demonstrate this Multi-Query Retriever.


In [23]:
from langchain_community.document_loaders import PyPDFLoader

In [24]:
loader = PyPDFLoader("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/ioch1wsxkfqgfLLgmd-6Rw/langchain-paper.pdf")
pdf_data = loader.load()

Let's take a look at the first page of this paper. This paper is talking about the LangChain framework.


In [25]:
pdf_data[1]

Document(metadata={'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2023-12-31T03:50:13+00:00', 'author': 'IEEE', 'moddate': '2023-12-31T03:52:06+00:00', 'title': 's8329 final', 'source': 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/ioch1wsxkfqgfLLgmd-6Rw/langchain-paper.pdf', 'total_pages': 6, 'page': 1, 'page_label': '2'}, page_content='LangChain helps us to unlock the ability to harness the \nLLM’s immense potential in tasks such as document analysis, \nchatbot development, code analysis, and countless other \napplications. Whether your desire is to unlock deeper natural \nlanguage understanding , enhance data, or circumvent \nlanguage barriers through translation, LangChain is ready to \nprovide the tools and programming support you need to do \nwithout it that it is not only difficult but also fresh for you. Its \ncore functionalities encompass: \n1. Context-Aware Capabilities: LangChain facilitates the \ndevelopment of applications that ar

Split the document and store the embeddings into a vector database.


In [26]:
# Split
chunks_pdf = text_splitter(pdf_data, 500, 20)

# VectorDB
try:
    ids = vectordb.get()["ids"]
    if ids:
        vectordb.delete(ids=ids)
except Exception:
    pass

vectordb = Chroma.from_documents(
    documents=chunks_pdf,
    embedding=hf_embedding()
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7684.00it/s]


The `MultiQueryRetriever` function from LangChain is used.


In [27]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

query = "What does the paper say about langchain?"

retriever = MultiQueryRetriever.from_llm(
    retriever=vectordb.as_retriever(), llm=llm()
)

Set logging for the queries.


In [28]:
import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

In [29]:
docs = retriever.invoke(query)
docs

[Document(id='6e329974-4e1c-4972-8a51-d0ce36145f5d', metadata={'producer': 'PyPDF', 'author': 'IEEE', 'creationdate': '2023-12-31T03:50:13+00:00', 'page_label': '6', 'source': 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/ioch1wsxkfqgfLLgmd-6Rw/langchain-paper.pdf', 'total_pages': 6, 'page': 5, 'creator': 'Microsoft Word', 'moddate': '2023-12-31T03:52:06+00:00', 'title': 's8329 final'}, page_content='[11] LangChain’s Large Language Model Chain,  \nhttps://python.langchain.com/docs/modules/chains/foundational/llm_c\nhain (accessed Nov. 29, 2023). \n[12] Streamlit, https://streamlit.io/ (accessed Nov. 29, 2023).'),
 Document(id='fb09f8cd-b004-4bab-a51a-3a8f903956bc', metadata={'creator': 'Microsoft Word', 'creationdate': '2023-12-31T03:50:13+00:00', 'producer': 'PyPDF', 'title': 's8329 final', 'author': 'IEEE', 'page': 0, 'page_label': '1', 'moddate': '2023-12-31T03:52:06+00:00', 'total_pages': 6, 'source': 'https://cf-courses-data.s3.us.cloud-object-storage.appdoma

From the log results, you can see that the LLM generated three additional queries from different perspectives based on the given query.

The returned results are the union of the results from each query.


#### Self-Querying Retriever


A Self-Querying Retriever, as the name suggests, has the ability to query itself. Specifically, given a natural language query, the retriever uses a query-constructing LLM chain to generate a structured query. It then applies this structured query to its underlying vector store. This enables the retriever to not only use the user-input query for semantic similarity comparison with the contents of stored documents but also to extract and apply filters based on the metadata of those documents.


The following code demonstrates how to use a Self-Querying Retriever.


In [30]:
from langchain_core.documents import Document
from langchain_classic.chains.query_constructor.base import AttributeInfo
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever

A couple of document pieces have been prepared where the `page_content` contains descriptions of movies, and the `meta_data` includes different attributes for each movie, such as `year`, `rating`, `genre`, and `director`. These attributes are crucial in the Self-Querying Retriever, as the LLM will use the metadata information to apply filters during the retrieval process.


In [50]:
docs = [
    Document(
        page_content="A bunch of scientists bring back dinosaurs and mayhem breaks loose",
        metadata={"year": 1993, "rating": 7.7, "genre": "science fiction"},
    ),
    Document(
        page_content="Leo DiCaprio gets lost in a dream within a dream within a dream within a ...",
        metadata={"year": 2010, "director": "Christopher Nolan", "rating": 8.2},
    ),
    Document(
        page_content="A psychologist / detective gets lost in a series of dreams within dreams within dreams and Inception reused the idea",
        metadata={"year": 2006, "director": "Satoshi Kon", "rating": 8.6},
    ),
    Document(
        page_content="A bunch of normal-sized women are supremely wholesome and some men pine after them",
        metadata={"year": 2019, "director": "Greta Gerwig", "rating": 8.3},
    ),
    Document(
        page_content="Toys come alive and have a blast doing so",
        metadata={"year": 1995, "genre": "animated"},
    ),
    Document(
        page_content="Three men walk into the Zone, three men walk out of the Zone",
        metadata={
            "year": 1979,
            "director": "Andrei Tarkovsky",
            "genre": "thriller",
            "rating": 9.9,
        },
    ),
]

Now you can instantiate your retriever. To do this, you'll need to provide some upfront information about the metadata fields that your documents support and a brief description of the document contents.


In [51]:
metadata_field_info = [
    AttributeInfo(
        name="genre",
        description="The genre of the movie. One of ['science fiction', 'comedy', 'drama', 'thriller', 'romance', 'action', 'animated']",
        type="string",
    ),
    AttributeInfo(
        name="year",
        description="The year the movie was released",
        type="integer",
    ),
    AttributeInfo(
        name="director",
        description="The name of the movie director",
        type="string",
    ),
    AttributeInfo(
        name="rating", description="A 1-10 rating for the movie", type="float"
    ),
]

Store the document's embeddings into a vector database.


In [52]:
vectordb = Chroma.from_documents(docs, hf_embedding())

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6453.52it/s]


Use the `SelfQueryRetriever`.


In [62]:
retriever = vectordb.as_retriever()

Now you can actually try using your retriever.


In [63]:
# This example only specifies a filter
retriever.invoke("I want to watch a movie rated higher than 8.5")

[Document(id='e79c063b-c42b-4d74-a719-3b4bd24d08ae', metadata={'director': 'Christopher Nolan', 'year': 2010, 'rating': 8.2}, page_content='Leo DiCaprio gets lost in a dream within a dream within a dream within a ...'),
 Document(id='06713454-4d86-4e9d-bc89-88fd30a8f22d', metadata={'rating': 8.2, 'year': 2010, 'director': 'Christopher Nolan'}, page_content='Leo DiCaprio gets lost in a dream within a dream within a dream within a ...'),
 Document(id='6e329974-4e1c-4972-8a51-d0ce36145f5d', metadata={'source': 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/ioch1wsxkfqgfLLgmd-6Rw/langchain-paper.pdf', 'author': 'IEEE', 'creator': 'Microsoft Word', 'moddate': '2023-12-31T03:52:06+00:00', 'page_label': '6', 'title': 's8329 final', 'page': 5, 'total_pages': 6, 'producer': 'PyPDF', 'creationdate': '2023-12-31T03:50:13+00:00'}, page_content='[11] LangChain’s Large Language Model Chain,  \nhttps://python.langchain.com/docs/modules/chains/foundational/llm_c\nhain (accessed No

In [64]:
# This example specifies a query and a filter
retriever.invoke("Has Greta Gerwig directed any movies about women")

[Document(id='7570421b-aa18-4ba7-9876-4d18ff25e5f5', metadata={'rating': 8.3, 'director': 'Greta Gerwig', 'year': 2019}, page_content='A bunch of normal-sized women are supremely wholesome and some men pine after them'),
 Document(id='18de657a-b27b-4f23-a4d6-2e119bd7eccd', metadata={'year': 2019, 'rating': 8.3, 'director': 'Greta Gerwig'}, page_content='A bunch of normal-sized women are supremely wholesome and some men pine after them'),
 Document(id='5fb5e237-9f17-479a-9b0f-c27e0f11bc29', metadata={'source': 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/ioch1wsxkfqgfLLgmd-6Rw/langchain-paper.pdf', 'moddate': '2023-12-31T03:52:06+00:00', 'page': 1, 'creationdate': '2023-12-31T03:50:13+00:00', 'page_label': '2', 'producer': 'PyPDF', 'creator': 'Microsoft Word', 'total_pages': 6, 'author': 'IEEE', 'title': 's8329 final'}, page_content='This approach with ChatModels opens the door to more \ndynamic and interactive conversations with the language \nmodel. It empowers 

When running the following cell, you might encounter some errors or blank content. This is because the LLM cannot get the answer at first. Don't worry; if you re-run it several times, you will get the answer.


In [65]:
# This example specifies a composite filter
retriever.invoke("What's a highly rated (above 8.5) science fiction film?")

[Document(id='e79c063b-c42b-4d74-a719-3b4bd24d08ae', metadata={'rating': 8.2, 'director': 'Christopher Nolan', 'year': 2010}, page_content='Leo DiCaprio gets lost in a dream within a dream within a dream within a ...'),
 Document(id='06713454-4d86-4e9d-bc89-88fd30a8f22d', metadata={'director': 'Christopher Nolan', 'year': 2010, 'rating': 8.2}, page_content='Leo DiCaprio gets lost in a dream within a dream within a dream within a ...'),
 Document(id='6e329974-4e1c-4972-8a51-d0ce36145f5d', metadata={'total_pages': 6, 'title': 's8329 final', 'producer': 'PyPDF', 'moddate': '2023-12-31T03:52:06+00:00', 'source': 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/ioch1wsxkfqgfLLgmd-6Rw/langchain-paper.pdf', 'page_label': '6', 'creationdate': '2023-12-31T03:50:13+00:00', 'creator': 'Microsoft Word', 'author': 'IEEE', 'page': 5}, page_content='[11] LangChain’s Large Language Model Chain,  \nhttps://python.langchain.com/docs/modules/chains/foundational/llm_c\nhain (accessed No

#### Parent Document Retriever


When splitting documents for retrieval, there are often conflicting desires:

1. You may want to have small documents so that their embeddings can most accurately reflect their meaning. If the documents are too long, the embeddings can lose meaning.
2. You want to have long enough documents so that the context of each chunk is retained.

The `ParentDocumentRetriever` strikes that balance by splitting and storing small chunks of data. During retrieval, it first fetches the small chunks but then looks up the parent IDs for those chunks and returns those larger documents.


In [66]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_text_splitters import CharacterTextSplitter
from langchain_classic.storage import InMemoryStore

In [67]:
# Set two splitters. One is with big chunk size (parent) and one is with small chunk size (child)
parent_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=20, separator='\n')
child_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20, separator='\n')

In [68]:
vectordb = Chroma(
    collection_name="split_parents", embedding_function=hf_embedding()
)
#vectordb = Chroma.from_documents(documents=chunks_pdf, embedding=watsonx_embedding())
# The storage layer for the parent documents
store = InMemoryStore()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5946.70it/s]


In [69]:
retriever = ParentDocumentRetriever(
    vectorstore=vectordb,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

In [ ]:
retriever.add_documents(txt_data)

zsh:1: command not found: ignorewarnings


Ignore the warnings as the documents before chunking have variable lengths.

These are the number of large chunks:


In [71]:
len(list(store.yield_keys()))

19

Let's make sure the underlying vector store still retrieves the small chunks.


In [72]:
sub_docs = vectordb.similarity_search("smoking policy")

In [73]:
print(sub_docs[0].page_content)

5.	Smoking Policy


Then, retrieve the relevant large chunk.


In [74]:
retrieved_docs = retriever.invoke("smoking policy")
print(retrieved_docs[0].page_content)

Cost Management: Keep personal phone usage separate from company accounts and reimburse the company for any personal charges on company-issued phones.
Compliance: Adhere to all pertinent laws and regulations concerning mobile phone usage, including those related to data protection and privacy.
Lost or Stolen Devices: Immediately report any lost or stolen mobile devices to the IT department or your supervisor.
Consequences: Non-compliance with this policy may lead to disciplinary actions, including the potential loss of mobile phone privileges.
The Mobile Phone Policy is aimed at promoting the responsible and secure use of mobile devices in line with legal and ethical standards. Every employee is expected to comprehend and abide by these guidelines. Regular reviews of the policy ensure its ongoing alignment with evolving technology and security best practices.
5.	Smoking Policy
